# Supervisor Pattern: Central Routing and Evidence Ownership

| Field | Value |
|---|---|
| Stage | Multi-agent RAG |
| Difficulty | Advanced |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
A supervisor owns routing and synthesis; specialists return bounded typed results rather than talking to the user.

## 30-Second Summary

A deterministic supervisor routes policy and arithmetic questions to dedicated specialists, checks their result schema, and abstains on unsupported intents.

## Why This Matters

Central routing makes ownership and budgets obvious when intents are stable, but the supervisor becomes a bottleneck if it performs specialist work itself.

## Scope

| Covers | Does not cover |
|---|---|
| Intent routing, specialist contracts, synthesis ownership, abstention | Learned router, parallel runtime, model-generated tool calls |


## Mental Model

```text
user -> supervisor -> one specialist -> typed result -> supervisor -> answer
```


In [1]:
POLICY = {"retention": {"value": 30, "unit": "days", "source": "policy-30"}}
queries = ["What is log retention?", "What is 6 times 7?", "What is tomorrow's weather?"]


## How It Works

The supervisor classifies each query, invokes one least-privilege specialist, validates the result type, and formats the final answer. Unsupported questions terminate without a handoff.


## Baseline

A route-everything-to-policy baseline answers only one of three intents correctly and risks hallucinating the others.


In [2]:
baseline_routes = ["policy" for _ in queries]
baseline_correct = sum(route == expected for route, expected in zip(baseline_routes, ["policy", "math", "abstain"]))
baseline_correct


1

## Technique Implementation

Routing is deliberately transparent so branch behavior and errors can be labeled and tested.


In [3]:
def route(query: str) -> str:
    lower = query.lower()
    if "retention" in lower: return "policy"
    if "times" in lower: return "math"
    return "abstain"

def dispatch(query: str) -> dict:
    branch = route(query)
    if branch == "policy": return {"branch": branch, **POLICY["retention"]}
    if branch == "math": return {"branch": branch, "value": 6 * 7, "unit": None, "source": "calculator"}
    return {"branch": branch, "reason": "unsupported intent"}

results = [dispatch(query) for query in queries]
results


[{'branch': 'policy', 'value': 30, 'unit': 'days', 'source': 'policy-30'},
 {'branch': 'math', 'value': 42, 'unit': None, 'source': 'calculator'},
 {'branch': 'abstain', 'reason': 'unsupported intent'}]

## Controlled Experiment

We score route accuracy and ensure only the supervisor formats user-facing answers.


In [4]:
expected = ["policy", "math", "abstain"]
route_accuracy = sum(result["branch"] == label for result, label in zip(results, expected)) / len(expected)
answers = [
    (f"{item['value']} {item['unit'] or ''} [{item['source']}]".strip() if item["branch"] != "abstain" else "I cannot answer from the available specialists.")
    for item in results
]
{"route_accuracy": route_accuracy, "answers": answers}


{'route_accuracy': 1.0,
 'answers': ['30 days [policy-30]',
  '42  [calculator]',
  'I cannot answer from the available specialists.']}

## Evaluation

The supervisor routes **3/3** labeled queries correctly; the policy-only baseline routes **1/3**. The fixture tests orchestration, not natural-language intent coverage.


In [5]:
assert baseline_correct == 1 and route_accuracy == 1.0
assert results[0]["source"] == "policy-30" and results[1]["value"] == 42
assert results[2] == {"branch": "abstain", "reason": "unsupported intent"}
print("Supervisor checks passed.")


Supervisor checks passed.


## Decision Guide

| Situation | Pattern |
|---|---|
| Stable mutually exclusive intents | Supervisor |
| Flexible peer collaboration | Network |
| Multiple managed teams | Hierarchy |
| Router confidence is low | Clarify or abstain |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Wrong specialist | Weak labels | Route test set/confidence gate |
| Supervisor does everything | Role leakage | Strict specialist contracts |
| Bottleneck | Serial central work | Parallelize independent calls |
| Hallucinated branch | No unsupported route | Explicit abstention |


## Production Notes

### Observability
Log route label/confidence, specialist, schema validation, latency, evidence IDs, and terminal reason.

### Safety and Guardrails
The supervisor cannot grant specialists extra permissions.

### Latency and Cost
Use a cheap router and bypass agents for deterministic operations.


## Practice

Add a retrieval-plus-calculation query and decide whether to invoke specialists sequentially or in parallel.

## Recall

Toggle - Recall: Who owns the final answer?
The supervisor.

Toggle - Recall: What is the supervisor's main risk?
A routing or availability bottleneck that affects every branch.

## Sources

- [LangChain subagents](https://docs.langchain.com/oss/python/langchain/multi-agent/subagents)
- Repository-owned synthetic routing fixture

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the labeled branch fixture | Add calibrated routing confidence |
